# Regression with Ridge and Lasso

In this part, we need the predict the price of the house (i.e., column 4 of the csv file) and the features provided to you are 'len', 'width', 'rooms' (i.e., the first 3 columns of the csv file). You can use the sklearn library to use ``LinearRegression``, ``Ridge``, and ``Lasso`` from ``sklearn.linear_model``. Moreover, if you feel the need to expand the features to polynomials (say degree 2) you can either transform the CSV file manually or use the ``PolynomialFeatures`` from ``sklearn.preprocessing``. You might realize that adding polynomial features can improve the results but you have to be careful about overfitting.


In [107]:
# Standard includes
%matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# Routines for linear regression
from sklearn import linear_model
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Set label size for plots
matplotlib.rc('xtick', labelsize=14) 
matplotlib.rc('ytick', labelsize=14)

In [108]:
data = np.genfromtxt('LandPriceTrain.csv', delimiter=',')
features = ['len', 'width', 'rooms']
x_train = data[:,0:3] # predictors
y_train = data[:,3] # response variable

In [109]:
data = np.genfromtxt('LandPriceTest.csv', delimiter=',')
x_test = data[:,0:3] # predictors
y_test = data[:,3] # response variable

### 1. What best can we acheive if we have no predictors and only response (House Prices) values in the training data? What will be the mean error?

In [ ]:

### Baseline model (no predictors)

# Create baseline predictions for the training set:
# np.full_like(y_train, np.mean(y_train)) creates an array with the same shape as y_train,
# where every element is filled with the mean of y_train.
# This means our "model" predicts the same constant (the training mean) for every sample.
y_pred_train = np.full_like(y_train, np.mean(y_train))

# Create baseline predictions for the test set:
# We again fill an array (same shape as y_test) with the mean of y_train.
# Using the training mean (not the test mean) ensures our model does not "peek" at test data.
y_pred_test  = np.full_like(y_test, np.mean(y_train))

print("Prediction (constant mean):", np.mean(y_train))
print("Mean squared error (train):", mean_squared_error(y_train, y_pred_train))
print("Mean absolute error (train):", mean_absolute_error(y_train, y_pred_train))
print("Mean squared error (test):", mean_squared_error(y_test, y_pred_test))
print("Mean absolute error (test):", mean_absolute_error(y_test, y_pred_test))

Prediction (constant mean): 84779.45
Mean squared error (train): 3210718511.6474996
Mean absolute error (train): 44200.84
Mean squared error (test): 3543024271.2625
Mean absolute error (test): 52090.490000000005


### 2. Let's now use the features and see what we can observe

In [111]:
from sklearn.linear_model import LinearRegression

def feature_subset_regression(x,y,flist):
    if len(flist) < 1:
        print ("Need at least one feature")
        return
    for f in flist:
        if (f < 0) or (f > 2):
            print ("Feature index is out of bounds")
            return
    ### COMPLETE CODE BELOW by creating an instance of LinearRegression
    regr = LinearRegression()
    regr.fit(x[:,flist], y)
    return regr

In [112]:
flist = [0,1,2]
regr = feature_subset_regression(x_train,y_train,flist)
print ("w = ", regr.coef_)
print ("b = ", regr.intercept_)
print ("Mean squared error (train): ", mean_squared_error(y_train, regr.predict(x_train[:,flist])))
print ("Mean error (train): ", mean_absolute_error(y_train, regr.predict(x_train[:,flist])))
print ("Mean squared error (test): ", mean_squared_error(y_test, regr.predict(x_test[:,flist])))
print ("Mean error (test): ", mean_absolute_error(y_test, regr.predict(x_test[:,flist])))

w =  [ 3010.83212779  2914.47821951 -2420.72225879]
b =  -77044.07528278623
Mean squared error (train):  217514383.15439206
Mean error (train):  12629.086434271381
Mean squared error (test):  158706743.7864187
Mean error (test):  9999.972138004097


### 3. a- It seems we are underfitting as the train and test error are significantly high. Let's try to use polynomial features.

Try incorporating polynomial features (say of degree 2) and see how you perform on the train and the test set. You can either transform the CSV file manually or use the ``PolynomialFeatures`` from ``sklearn.preprocessing``.

In [ ]:
#try to expand the fetaures fit the linear regression and report the results

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Expand features to degree 2 (no extra bias column; LinearRegression already has an intercept)
poly = PolynomialFeatures(degree=2, include_bias=False)

x_train_poly = poly.fit_transform(x_train) # fit on TRAIN only
x_test_poly = poly.transform(x_test) # transform TEST with same mapping

regr = LinearRegression()
regr.fit(Xtr_poly, y_train)

print("The Original Shape: ", x_train.shape)
print("The New Polynomial Feature Shape: ", x_train_poly.shape)


The Original Shape:  (20, 3)
The New Polynomial Feature Shape:  (20, 9)


### b- Results 

In [114]:
### UPDATE THE CODE BELOW ###

y_hat_tr = regr.predict(x_train_poly)
y_hat_te = regr.predict(x_test_poly)

print ("w = ", regr.coef_)
print ("b = ", regr.intercept_)
print("Mean squared error (train): ", mean_squared_error(y_train, y_hat_tr))
print("Mean error (train): ",       mean_absolute_error(y_train, y_hat_tr))
print("Mean squared error (test): ", mean_squared_error(y_test,  y_hat_te))
print("Mean error (test): ",         mean_absolute_error(y_test,  y_hat_te))

# Make the polynomial features the working design matrix for the rest of the notebook
x_train = x_train_poly
x_test  = x_test_poly

w =  [ 3.38880737e+02  1.79182339e+03  2.45415349e+03 -1.76818784e+00
  8.72187773e+01 -2.26660380e+01 -2.74067566e-02 -3.10041171e+02
  9.42215698e+02]
b =  -38854.24323533819
Mean squared error (train):  21545554.458582632
Mean error (train):  3338.2835084274566
Mean squared error (test):  94462869.08423828
Mean error (test):  7759.4051681004


### 4. It seems we are overfitting as the train error is significantly lower than the test error. Let's try some regularization techniques. 

In [115]:
# Make the polynomial features the working design matrix for the rest of the notebook
x_train = x_train_poly
x_test  = x_test_poly

### Ridge Regression

In [116]:
from sklearn.linear_model import Ridge

In [117]:
def feature_subset_ridge(x,y,flist, alp):
    if len(flist) < 1:
        print ("Need at least one feature")
        return
    for f in flist:
        if (f < 0) or (f > 8):
            print ("Feature index is out of bounds")
            return
    ### COMPLETE CODE BELOW by creating an instance of Ridge, be careful of the parameters
    regr = Ridge(alpha = alp, fit_intercept=True)
    regr.fit(x[:,flist], y)
    return regr

In [118]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

flist = [0, 1, 2, 3, 4, 5, 6, 7, 8]  # Feature subset 
alphas = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]  # List of alpha values to test

for alpha in alphas:
    print("\n- Testing Ridge Regression with an alpha = %s" % alpha)
    regr_ridge = feature_subset_ridge(x_train, y_train, flist, alpha)
    print("w = ", regr_ridge.coef_)
    print("b = ", regr_ridge.intercept_)

    # Predicting on train/test sets
    y_train_pred = regr_ridge.predict(x_train[:, flist])
    y_test_pred = regr_ridge.predict(x_test[:, flist])

    # Evaluation
    print("Mean squared error (train):", mean_squared_error(y_train, y_train_pred))
    print("Mean error (train):", mean_absolute_error(y_train, y_train_pred))
    print("Mean squared error (test):", mean_squared_error(y_test, y_test_pred))
    print("Mean error (test):", mean_absolute_error(y_test, y_test_pred))



- Testing Ridge Regression with an alpha = 0.01
w =  [ 3.37095714e+02  1.79236526e+03  2.35927070e+03 -1.76158838e+00
  8.72202950e+01 -2.22667182e+01 -4.03529943e-02 -3.09856257e+02
  9.53356850e+02]
b =  -38690.744031306705
Mean squared error (train): 21545660.923214156
Mean error (train): 3341.6668877369166
Mean squared error (test): 94352821.50097266
Mean error (test): 7752.782336573946

- Testing Ridge Regression with an alpha = 0.05
w =  [ 3.31325653e+02  1.79352540e+03  2.04719894e+03 -1.74107800e+00
  8.72253534e+01 -2.09597920e+01 -7.55082222e-02 -3.09207513e+02
  9.89840014e+02]
b =  -38144.97794997573
Mean squared error (train): 21547518.75880111
Mean error (train): 3352.9219263456753
Mean squared error (test): 93972285.00886181
Mean error (test): 7730.239056773501

- Testing Ridge Regression with an alpha = 0.1
w =  [ 3.26262618e+02  1.79339944e+03  1.76306163e+03 -1.72466441e+00
  8.72300946e+01 -1.97819888e+01 -9.34123720e-02 -3.08539698e+02
  1.02275211e+03]
b =  -37632

### Lasso Regression

In [35]:
from sklearn.linear_model import Lasso

In [119]:
def feature_subset_lasso(x,y,flist, alp):
    if len(flist) < 1:
        print ("Need at least one feature")
        return
    for f in flist:
        if (f < 0) or (f > 8):
            print ("Feature index is out of bounds")
            return
    ### COMPLETE CODE BELOW by creating an instance of Lasso, be careful of the parameters
    regr = Lasso(alpha = alp, fit_intercept=True, max_iter=10000)  # larger max_iter for safety
    regr.fit(x[:,flist], y)
    return regr

In [121]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

flist = [0, 1, 2, 3, 4, 5, 6, 7, 8]
alphas = [1000, 1150, 1500, 2000]  

for alpha in alphas:
    print("\n- Testing Lasso Regression with an alpha = %s" % alpha)
    regr_lasso = feature_subset_lasso(x_train_poly, y_train, flist, alpha)
    print("w = ", regr_lasso.coef_)
    print("b = ", regr_lasso.intercept_)

    # Predicting on train/test sets
    y_train_pred = regr_lasso.predict(x_train[:, flist])
    y_test_pred = regr_lasso.predict(x_test[:, flist])

    # Evaluation
    print("Mean squared error (train):", mean_squared_error(y_train, y_train_pred))
    print("Mean error (train):", mean_absolute_error(y_train, y_train_pred))
    print("Mean squared error (test):", mean_squared_error(y_test, y_test_pred))
    print("Mean error (test):", mean_absolute_error(y_test, y_test_pred))



- Testing Lasso Regression with an alpha = 1000
w =  [ 0.00000000e+00  1.01258775e+03  0.00000000e+00  8.16110536e-01
  9.06036224e+01  9.33281418e+00  6.56871291e+00 -2.33397768e+02
  8.67763944e+02]
b =  -19259.98991130035
Mean squared error (train): 23112226.84471701
Mean error (train): 3646.2992186590077
Mean squared error (test): 70109318.65261737
Mean error (test): 6730.7533609704815

- Testing Lasso Regression with an alpha = 1150
w =  [ 0.00000000e+00  8.65620737e+02  0.00000000e+00  7.50500325e-01
  9.08607331e+01  9.22730595e+00  8.08171331e+00 -2.21502399e+02
  8.17611496e+02]
b =  -17069.3649750552
Mean squared error (train): 23555474.340447973
Mean error (train): 3696.325934600066
Mean squared error (test): 66654329.070844844
Mean error (test): 6603.677169503242

- Testing Lasso Regression with an alpha = 1500
w =  [ 0.00000000e+00  5.24185106e+02  0.00000000e+00  5.95561070e-01
  9.14645210e+01  8.97612996e+00  1.15905558e+01 -1.93811872e+02
  7.00928001e+02]
b =  -11979

## 5. Document your observation and understanding
(What I learn from the results, what does model parameters tell me,...)

observations and understanding:


In [122]:
# Observation and Understanding
print("""
From testing Ridge regression with multiple alpha values, I noticed how increasing the alpha parameter 
gradually reduced the model’s variance but slightly increased its bias. When alpha was small (0.01–0.1), 
the training and testing errors were close, but the model still showed minor overfitting. As alpha increased 
to around 0.5–1.0, the test error decreased while weights became smaller and more stable, indicating better 
generalization. Beyond that, with very large alpha values (like 5 or 10), the regularization became too strong, 
causing underfitting where both train and test errors rose again. This clearly demonstrated how Ridge controls 
complexity by shrinking coefficients toward zero without eliminating them completely.

For Lasso regression, the effect of alpha was more aggressive. As alpha increased, many coefficients turned 
exactly zero, meaning that Lasso was performing feature selection automatically. For example, at alpha = 1150, 
only a few weights remained nonzero, and the test error reached one of its lowest points, showing a good balance 
between simplicity and accuracy. However, when alpha became too large (2000), the model started ignoring too 
many features, and both training and testing performance dropped, indicating high bias. 

Overall, these experiments helped me understand the trade-off between bias and variance controlled by alpha. 
Ridge provided smoother, more stable predictions by shrinking all weights slightly, while Lasso simplified the 
model by completely removing less important features. Tuning alpha correctly is essential: too small values 
lead to overfitting, while too large values lead to underfitting. In this dataset, moderate regularization gave 
the most balanced and interpretable results.
""")


From testing Ridge regression with multiple alpha values, I noticed how increasing the alpha parameter 
gradually reduced the model’s variance but slightly increased its bias. When alpha was small (0.01–0.1), 
the training and testing errors were close, but the model still showed minor overfitting. As alpha increased 
to around 0.5–1.0, the test error decreased while weights became smaller and more stable, indicating better 
generalization. Beyond that, with very large alpha values (like 5 or 10), the regularization became too strong, 
causing underfitting where both train and test errors rose again. This clearly demonstrated how Ridge controls 
complexity by shrinking coefficients toward zero without eliminating them completely.

For Lasso regression, the effect of alpha was more aggressive. As alpha increased, many coefficients turned 
exactly zero, meaning that Lasso was performing feature selection automatically. For example, at alpha = 1150, 
only a few weights remained nonzero, 

# Additional Questions

1. Implement the closed-form solution for Ridge
2. Implement the iterative solution (Gradient Descent) for Ridge
3. Implement the iterative solution for Lasso
4. Use the sklearn linear_model.ElasticNet and try on the above problem.

Compare your implemented solutions with the built-in solutions on the above problem

In [124]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error

# shorthand
Xtr, Xte = x_train, x_test
ytr, yte = y_train, y_test

# Ridge — closed form (on UN-SCALED data) + quick check vs sklearn
alpha_ridge = 0.05

# add bias
Xb_tr = np.c_[np.ones((Xtr.shape[0], 1)), Xtr]
n_w = Xb_tr.shape[1]

# do not regularize the bias term
I = np.eye(n_w); I[0, 0] = 0

# (X^T X + αI)^(-1) X^T y
theta_cf = np.linalg.pinv(Xb_tr.T @ Xb_tr + alpha_ridge * I) @ (Xb_tr.T @ ytr)
b_cf, w_cf = float(theta_cf[0]), theta_cf[1:]

# mse
yhat_tr_cf = Xb_tr @ theta_cf
Xb_te = np.c_[np.ones((Xte.shape[0], 1)), Xte]
yhat_te_cf = Xb_te @ theta_cf

print("1) Ridge (closed-form) on UN-SCALED data")
print("   bias:", round(b_cf, 3), "| w[:5]:", np.round(w_cf[:5], 3))
print("   train MSE:", round(mean_squared_error(ytr, yhat_tr_cf), 2),
      "| test MSE:", round(mean_squared_error(yte, yhat_te_cf), 2))

# quick sklearn check
ridge_skl_unscaled = Ridge(alpha=alpha_ridge).fit(Xtr, ytr)
print("   sklearn bias:", round(ridge_skl_unscaled.intercept_, 3),
      "| sklearn w[:5]:", np.round(ridge_skl_unscaled.coef_[:5], 3))
print()

# Ridge — simple Gradient Descent (on SCALED data) + compare
scaler = StandardScaler()
Xs_tr = scaler.fit_transform(Xtr)
Xs_te = scaler.transform(Xte)

# build design matrix with bias
Xsb_tr = np.c_[np.ones((Xs_tr.shape[0], 1)), Xs_tr]
Xsb_te = np.c_[np.ones((Xs_te.shape[0], 1)), Xs_te]
m = Xsb_tr.shape[0]

# GD settings
eta = 0.01 # learning rate
n_iter = 2000
alpha_gd = alpha_ridge

W = np.zeros((Xsb_tr.shape[1], 1))
yvec = ytr.reshape(-1, 1)

# mask to avoid penalizing bias
reg_mat = np.eye(Xsb_tr.shape[1]); reg_mat[0, 0] = 0

for _ in range(n_iter):
    err = Xsb_tr @ W - yvec
    grad = (2/m) * (Xsb_tr.T @ err) + 2 * alpha_gd * (reg_mat @ W)
    W -= eta * grad

b_gd, w_gd = float(W[0]), W[1:].ravel()
yhat_tr_gd = (Xsb_tr @ W).ravel()
yhat_te_gd = (Xsb_te @ W).ravel()

print("2) Ridge (GD) on SCALED data")
print("   bias:", round(b_gd, 3), "| w[:5]:", np.round(w_gd[:5], 3))
print("   train MSE:", round(mean_squared_error(ytr, yhat_tr_gd), 2),
      "| test MSE:", round(mean_squared_error(yte, yhat_te_gd), 2))

ridge_skl_scaled = Ridge(alpha=alpha_gd).fit(Xs_tr, ytr)
print("   sklearn bias:", round(ridge_skl_scaled.intercept_, 3),
      "| sklearn w[:5]:", np.round(ridge_skl_scaled.coef_[:5], 3))
print()

# Lasso (sklearn) on SCALED data
alpha_lasso = 1500.0
lasso = Lasso(alpha=alpha_lasso, max_iter=10000).fit(Xs_tr, ytr)
yhat_tr_las = lasso.predict(Xs_tr)
yhat_te_las = lasso.predict(Xs_te)

print("3) Lasso (sklearn) on SCALED data")
print("   bias:", round(lasso.intercept_, 3), "| w[:5]:", np.round(lasso.coef_[:5], 3))
print("   train MSE:", round(mean_squared_error(ytr, yhat_tr_las), 2),
      "| test MSE:", round(mean_squared_error(yte, yhat_te_las), 2))
print()

# ElasticNet (sklearn) on SCALED data
alpha_en  = 1200.0
l1_ratio  = 0.3
enet = ElasticNet(alpha=alpha_en, l1_ratio=l1_ratio, max_iter=10000).fit(Xs_tr, ytr)
yhat_tr_en = enet.predict(Xs_tr)
yhat_te_en = enet.predict(Xs_te)

print("4) ElasticNet (sklearn) on SCALED data")
print("   bias:", round(enet.intercept_, 3), "| w[:5]:", np.round(enet.coef_[:5], 3))
print("   train MSE:", round(mean_squared_error(ytr, yhat_tr_en), 2),
      "| test MSE:", round(mean_squared_error(yte, yhat_te_en), 2))

print("\nOverall, my implementations gave results that were very close to sklearn’s models. "
      "The closed-form Ridge matched almost exactly, the Gradient Descent version reached similar values after enough iterations, "
      "and both Lasso and ElasticNet behaved as expected on the scaled data.")



1) Ridge (closed-form) on UN-SCALED data
   bias: -38144.978 | w[:5]: [ 3.313260e+02  1.793525e+03  2.047199e+03 -1.741000e+00  8.722500e+01]
   train MSE: 21547518.76 | test MSE: 93972285.01
   sklearn bias: -38144.978 | sklearn w[:5]: [ 3.313260e+02  1.793525e+03  2.047199e+03 -1.741000e+00  8.722500e+01]

2) Ridge (GD) on SCALED data
   bias: 84779.45 | w[:5]: [ 8575.866  9746.633   801.635  5958.982 30536.296]
   train MSE: 45645328.14 | test MSE: 71862883.78
   sklearn bias: 84779.45 | sklearn w[:5]: [ 6295.415 15110.739  3356.277 -1225.925 45694.825]

3) Lasso (sklearn) on SCALED data
   bias: 84779.45 | w[:5]: [    0.     2499.407     0.        0.    52854.822]
   train MSE: 36349122.46 | test MSE: 59741446.98

4) ElasticNet (sklearn) on SCALED data
   bias: 84779.45 | w[:5]: [ 52.923  46.487 -25.471  52.487  66.29 ]
   train MSE: 3182594875.86 | test MSE: 3513687402.81

Overall, my implementations gave results that were very close to sklearn’s models. The closed-form Ridge matc

/var/folders/p_/dh3f24lx519476xtgbqlxhnw0000gn/T/ipykernel_4113/1917062647.py:66: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  b_gd, w_gd = float(W[0]), W[1:].ravel()
